# Chapter 6: Finetuning for Text Classification

Reference: `LLMs-from-scratch/ch06/`

**Goals:**
- Load pretrained model weights
- Replace output head for classification
- Finetune on a labeled dataset (spam detection)
- Evaluate classification accuracy

In [2]:
import sys; sys.path.append('..')
import torch
import tiktoken
import numpy as np
from src.models import GPTModel
from src.config import GPT_CONFIG_124M
import src.gpt_download as gpt_download
from src.models import generate_text_simple
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn as nn
import urllib.request
import zipfile
import os
from pathlib import Path
import pandas as pd
from torch.utils.data import Dataset, DataLoader

## 1. Load Pretrained Weights

In [3]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [4]:
# 1. Download real GPT-2 124M weights
settings, params = gpt_download.download_and_load_gpt2(
    model_size="124M", models_dir="../gpt2"
)

# 2. Load into our model
model = GPTModel(GPT_CONFIG_124M)
load_weights_into_gpt(model, params)
model.to(device)
model.eval()

# 3. Test generation - should produce fluent English now
tokenizer = tiktoken.get_encoding("gpt2")
text = "The weather today is"
encoded = torch.tensor(tokenizer.encode(text), device=device).unsqueeze(0)
with torch.no_grad():
    out = generate_text_simple(model, encoded, max_new_tokens=20,context_size=GPT_CONFIG_124M["context_length"])
print(tokenizer.decode(out[0].tolist()))

File already exists and is up-to-date: ../gpt2\124M\checkpoint
File already exists and is up-to-date: ../gpt2\124M\encoder.json
File already exists and is up-to-date: ../gpt2\124M\hparams.json
File already exists and is up-to-date: ../gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ../gpt2\124M\model.ckpt.index
File already exists and is up-to-date: ../gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: ../gpt2\124M\vocab.bpe
The weather today is very good, and we're going to be able to get some good weather tomorrow.

"


## 2. Classification Head

In [5]:
class GPTClassifier(nn.Module):
    def __init__(self, gpt_model, num_classes=2):
        super().__init__()
        self.gpt = gpt_model
        self.class_head = nn.Linear(gpt_model.out_head.in_features, num_classes)

    def forward(self, x):
        with torch.no_grad():
            x = self.gpt.tok_emb(x)
            x = x + self.gpt.pos_emb(torch.arange(x.shape[1], device=x.device))
            x = self.gpt.drop_emb(x)
            x = self.gpt.trf_blocks(x)
            x = self.gpt.final_norm(x)
        last_token = x[:, -1, :]
        return self.class_head(last_token)

# Test
classifier = GPTClassifier(model, num_classes=2).to(device)
sample = torch.randint(0, 50256, (2, 10)).to(device)
out = classifier(sample)
print(f"Input shape:  {sample.shape}")
print(f"Output shape: {out.shape}")  # should be [2, 2]

Input shape:  torch.Size([2, 10])
Output shape: torch.Size([2, 2])


## 3. Prepare Classification Dataset

In [6]:
# 1. Download & extract SMS Spam Collection
data_dir = Path("../data")
zip_path = data_dir / "sms_spam.zip"
extracted_path = data_dir / "sms_spam"
data_file_path = data_dir / "sms_spam.tsv"

if not data_file_path.exists():
    url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(extracted_path)
    os.rename(extracted_path / "SMSSpamCollection", data_file_path)
    print("Downloaded and extracted.")
else:
    print("File already exists.")

# 2. Load as DataFrame
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df["Label"] = df["Label"].map({"ham": 0, "spam": 1})
print(f"Total samples: {len(df)}")
print(f"Spam: {df['Label'].sum()}, Ham: {(1 - df['Label']).sum()}")

# 3. Balance the dataset
spam = df[df["Label"] == 1]
ham = df[df["Label"] == 0].sample(len(spam), random_state=123)
df_balanced = pd.concat([spam, ham]).sample(frac=1, random_state=123).reset_index(drop=True)
print(f"Balanced: {len(df_balanced)} samples")

# 4. Split
train_frac, val_frac = 0.7, 0.15
train_end = int(len(df_balanced) * train_frac)
val_end = train_end + int(len(df_balanced) * val_frac)
train_df = df_balanced[:train_end]
val_df = df_balanced[train_end:val_end]
test_df = df_balanced[val_end:]
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

File already exists.
Total samples: 5572
Spam: 747, Ham: 4825
Balanced: 1494 samples
Train: 1045, Val: 224, Test: 225


In [7]:
class SpamDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=None, pad_token_id=50256):
        self.data = df
        self.labels = df["Label"].values
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad_token_id = pad_token_id

        # Tokenize all texts
        self.encoded_texts = [
            tokenizer.encode(text, allowed_special={"<|endoftext|>"})
            for text in df["Text"]
        ]

        # Determine max length if not provided
        if max_length is None:
            self.max_length = max(len(enc) for enc in self.encoded_texts)

        # Pad all sequences to max_length
        self.encoded_texts = [
            enc + [pad_token_id] * (self.max_length - len(enc))
            for enc in self.encoded_texts
        ]

    def __getitem__(self, idx):
        tokens = torch.tensor(self.encoded_texts[idx])
        label = torch.tensor(self.labels[idx])
        return tokens, label

    def __len__(self):
        return len(self.data)

# Create datasets and dataloaders
tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset(train_df, tokenizer)
val_dataset = SpamDataset(val_df, tokenizer, max_length=train_dataset.max_length)
test_dataset = SpamDataset(test_df, tokenizer, max_length=train_dataset.max_length)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, drop_last=False)

print(f"Max sequence length: {train_dataset.max_length}")
for x, y in train_loader:
    print(f"Batch input shape: {x.shape}")
    print(f"Batch label shape: {y.shape}")
    break

Max sequence length: 120
Batch input shape: torch.Size([8, 120])
Batch label shape: torch.Size([8])


## 4. Finetuning Loop

In [8]:
def calc_accuracy_loader(data_loader, model, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for input_batch, target_batch in data_loader:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            logits = model(input_batch)
            predicted = torch.argmax(logits, dim=-1)
            correct += (predicted == target_batch).sum().item()
            total += target_batch.size(0)
    return correct / total

def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    global_step = -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            optimizer.zero_grad()
            logits = model(input_batch)
            loss = nn.functional.cross_entropy(logits, target_batch)
            loss.backward()
            optimizer.step()
            global_step += 1

            if global_step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss = nn.functional.cross_entropy(
                        model(input_batch), target_batch).item()
                    val_losses_batch = [
                        nn.functional.cross_entropy(
                            model(vb.to(device)), lb.to(device)).item()
                        for vb, lb in val_loader
                    ]
                train_losses.append(train_loss)
                val_losses.append(sum(val_losses_batch) / len(val_losses_batch))
                train_accs.append(calc_accuracy_loader(train_loader, model, device))
                val_accs.append(calc_accuracy_loader(val_loader, model, device))
                print(f"Ep {epoch+1} (Step {global_step:03d}): "
                      f"Train loss {train_losses[-1]:.3f} | Val loss {val_losses[-1]:.3f} | "
                      f"Train acc {train_accs[-1]:.3f} | Val acc {val_accs[-1]:.3f}")

    return train_losses, val_losses, train_accs, val_accs

# Freeze GPT backbone, only train classifier head
for param in model.parameters():
    param.requires_grad = False

# Re-create classifier (so it gets the fresh model reference)
classifier = GPTClassifier(model, num_classes=2).to(device)

# Only classifier head params are trainable
optimizer = torch.optim.AdamW(classifier.class_head.parameters(), lr=5e-5)

print(f"Trainable params: {sum(p.numel() for p in classifier.parameters() if p.requires_grad):,}")
print(f"Total params: {sum(p.numel() for p in classifier.parameters()):,}")

train_losses, val_losses, train_accs, val_accs = train_classifier_simple(
    classifier, train_loader, val_loader, optimizer, device,
    num_epochs=5, eval_freq=25)

Trainable params: 1,538
Total params: 163,038,722
Ep 1 (Step 000): Train loss 0.212 | Val loss 0.968 | Train acc 0.505 | Val acc 0.482
Ep 1 (Step 025): Train loss 0.672 | Val loss 0.739 | Train acc 0.504 | Val acc 0.482
Ep 1 (Step 050): Train loss 0.682 | Val loss 0.686 | Train acc 0.533 | Val acc 0.567
Ep 1 (Step 075): Train loss 0.678 | Val loss 0.684 | Train acc 0.500 | Val acc 0.540
Ep 1 (Step 100): Train loss 0.693 | Val loss 0.681 | Train acc 0.678 | Val acc 0.688
Ep 1 (Step 125): Train loss 0.717 | Val loss 0.681 | Train acc 0.537 | Val acc 0.509
Ep 2 (Step 150): Train loss 0.580 | Val loss 0.695 | Train acc 0.494 | Val acc 0.518
Ep 2 (Step 175): Train loss 0.667 | Val loss 0.674 | Train acc 0.503 | Val acc 0.536
Ep 2 (Step 200): Train loss 0.695 | Val loss 0.677 | Train acc 0.506 | Val acc 0.491
Ep 2 (Step 225): Train loss 0.664 | Val loss 0.668 | Train acc 0.539 | Val acc 0.585
Ep 2 (Step 250): Train loss 0.661 | Val loss 0.666 | Train acc 0.790 | Val acc 0.790
Ep 3 (Step 275)

## 5. Evaluation

In [9]:
# Test accuracy
test_acc = calc_accuracy_loader(test_loader, classifier, device)
print(f"Test accuracy: {test_acc:.3f} ({test_acc*100:.1f}%)")

# Show some predictions
tokenizer = tiktoken.get_encoding("gpt2")
classifier.eval()
samples_to_show = 5

with torch.no_grad():
    for i, (input_batch, target_batch) in enumerate(test_loader):
        if i >= 1:  # show from first batch only
            break
        input_batch = input_batch.to(device)
        logits = classifier(input_batch)
        preds = torch.argmax(logits, dim=-1)

        for j in range(min(samples_to_show, len(input_batch))):
            text = tokenizer.decode(input_batch[j].tolist())
            print(f"\n{'─'*50}")
            print(f"Text: {text[:100]}...")
            print(f"True:  {'SPAM' if target_batch[j] == 1 else 'ham'}")
            print(f"Pred:  {'SPAM' if preds[j] == 1 else 'ham'}")

Test accuracy: 0.747 (74.7%)

──────────────────────────────────────────────────
Text: You have won a Nokia 7250i. This is what you get when you win our FREE auction. To take part send No...
True:  SPAM
Pred:  SPAM

──────────────────────────────────────────────────
Text: Thats cool. i am a gentleman and will treat you with dignity and respect.<|endoftext|><|endoftext|><...
True:  ham
Pred:  SPAM

──────────────────────────────────────────────────
Text: ILL B DOWN SOON<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endof...
True:  ham
Pred:  ham

──────────────────────────────────────────────────
Text: Do you want a new Video phone? 600 anytime any network mins 400 Inclusive Video calls AND downloads ...
True:  SPAM
Pred:  SPAM

──────────────────────────────────────────────────
Text: Your 2004 account for 07XXXXXXXXX shows 786 unredeemed points. To claim call 08719181259 Identifier ...
True:  SPAM
Pred:  SPAM


In [12]:
class GPTClassifierFull(nn.Module):
    def __init__(self, gpt_model, num_classes=2):
        super().__init__()
        self.gpt = gpt_model
        self.class_head = nn.Linear(gpt_model.out_head.in_features, num_classes)

    def forward(self, x):
        x = self.gpt.tok_emb(x)
        x = x + self.gpt.pos_emb(torch.arange(x.shape[1], device=x.device))
        x = self.gpt.drop_emb(x)
        x = self.gpt.trf_blocks(x)
        x = self.gpt.final_norm(x)
        last_token = x[:, -1, :]
        return self.class_head(last_token)

In [13]:
# Unfreeze all GPT layers
for param in model.parameters():
    param.requires_grad = True

classifier_full = GPTClassifierFull(model, num_classes=2).to(device)

optimizer_full = torch.optim.AdamW(classifier_full.parameters(), lr=1e-5)

train_losses2, val_losses2, train_accs2, val_accs2 = train_classifier_simple(
    classifier_full, train_loader, val_loader, optimizer_full, device,
    num_epochs=3, eval_freq=25)

Ep 1 (Step 000): Train loss 2.519 | Val loss 3.621 | Train acc 0.506 | Val acc 0.482
Ep 1 (Step 025): Train loss 0.625 | Val loss 0.589 | Train acc 0.513 | Val acc 0.549
Ep 1 (Step 050): Train loss 0.681 | Val loss 0.404 | Train acc 0.820 | Val acc 0.830
Ep 1 (Step 075): Train loss 0.326 | Val loss 0.361 | Train acc 0.856 | Val acc 0.848
Ep 1 (Step 100): Train loss 0.579 | Val loss 0.485 | Train acc 0.816 | Val acc 0.830
Ep 1 (Step 125): Train loss 0.322 | Val loss 0.324 | Train acc 0.881 | Val acc 0.871
Ep 2 (Step 150): Train loss 0.554 | Val loss 0.677 | Train acc 0.795 | Val acc 0.763
Ep 2 (Step 175): Train loss 0.249 | Val loss 0.429 | Train acc 0.815 | Val acc 0.812
Ep 2 (Step 200): Train loss 0.232 | Val loss 0.372 | Train acc 0.843 | Val acc 0.839
Ep 2 (Step 225): Train loss 0.103 | Val loss 0.376 | Train acc 0.863 | Val acc 0.853
Ep 2 (Step 250): Train loss 0.105 | Val loss 0.263 | Train acc 0.907 | Val acc 0.911
Ep 3 (Step 275): Train loss 0.678 | Val loss 0.280 | Train acc 0.

In [15]:
test_acc2 = calc_accuracy_loader(test_loader, classifier_full, device)
print(f"Full finetune test accuracy: {test_acc2:.3f} ({test_acc2*100:.1f}%)")
print(f"Head-only was: 74.7%")
print(f"Improvement:    {test_acc2*100 - 74.7:.1f}%")

Full finetune test accuracy: 0.973 (97.3%)
Head-only was: 74.7%
Improvement:    22.6%
